# Moondream2 Gaze Evaluation

In [ ]:
import json, os
from datetime import datetime, timezone
from pathlib import Path
from datasets import load_dataset
from transformers import AutoModelForCausalLM
import torch
from tqdm.auto import tqdm
import numpy as np


RESULTS_PATH = Path("./MoondreamEval.json")
REVISION = "2025-01-09"
SAVE_INTERVAL = 50
os.environ["HF_TOKEN"] = "deleted"

In [4]:
dataset = load_dataset("grow-ai-like-a-child/gaze-referent-stimuli", split="train")
print(f"Total stimuli in dataset: {len(dataset)}")

Total stimuli in dataset: 2273


In [ ]:
existing_results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}
evaluated_ids = set(existing_results.keys())
to_evaluate = [entry for entry in dataset if entry["stimulus_id"] not in evaluated_ids]
print(f"Already evaluated: {len(evaluated_ids)}, Remaining: {len(to_evaluate)}")

model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    dtype=torch.float16,
    device_map={"": "mps"},
    revision=REVISION,
    low_cpu_mem_usage=True
)
# conda install -c conda-forge libvips

for i, entry in enumerate(tqdm(to_evaluate, desc="Evaluating")):
    face = {
        "x_min": entry["face_x_min"],
        "y_min": entry["face_y_min"],
        "x_max": entry["face_x_max"],
        "y_max": entry["face_y_max"]
    }
    
    gaze = model.detect_gaze(
        entry["image"],
        eye=None,
        face=face,
        unstable_settings={"prioritize_accuracy": True, "force_detect": True}
    )["gaze"]
    
    existing_results[entry["stimulus_id"]] = {
        "gaze": {"x": gaze["x"], "y": gaze["y"]} if gaze else None,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "revision": REVISION
    }
    
    if (i + 1) % SAVE_INTERVAL == 0:
        RESULTS_PATH.write_text(json.dumps(existing_results, indent=2))
    
RESULTS_PATH.write_text(json.dumps(existing_results, indent=2))

/Users/zory/miniforge3/envs/gaze/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total stimuli in dataset: 2273


KeyboardInterrupt: 

In [6]:

# Load results and dataset for analysis
results = json.loads(RESULTS_PATH.read_text())

# Calculate L2 distance between predicted gaze and ground truth label position
distances = []
missing_gaze = 0

for entry in dataset:
    stim_id = str(entry["stimulus_id"])
    if stim_id not in results:
        continue
    
    result = results[stim_id]
    if result["gaze"] is None:
        missing_gaze += 1
        continue
    
    pred_x, pred_y = result["gaze"]["x"], result["gaze"]["y"]
    
    # Find ground truth position based on label
    label = entry["label"]
    candidates = entry["candidates"].split("+")
    
    # Find which option index matches the label
    gt_x, gt_y = None, None
    for i, candidate in enumerate(candidates, start=1):
        if candidate == label:
            gt_x = entry[f"option{i}_x"]
            gt_y = entry[f"option{i}_y"]
            break
    
    if gt_x is None or gt_x < 0:
        continue
    
    #dist = np.sqrt((pred_x - gt_x)**2 + (pred_y - gt_y)**2)
    dist = np.abs(pred_x - gt_x)
    distances.append(dist)

print(f"Total evaluated: {len(results)}")
print(f"Missing gaze predictions: {missing_gaze}")
print(f"Valid comparisons: {len(distances)}")
print(f"Average L2 distance: {np.mean(distances):.4f}")
print(f"Std L2 distance: {np.std(distances):.4f}")
print(f"Median L2 distance: {np.median(distances):.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'MoondreamEval.json'